In [ ]:
import sys
import os




project_path = r"C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder"
if project_path not in sys.path:
    sys.path.append(project_path)

import track_builder as tb

import pandas as pd
import track_builder as tb

import os
from dotenv import load_dotenv

load_dotenv()

saving_path = os.getenv("SAVING")
output_path = os.path.join(saving_path, "tracks")
os.makedirs(output_path, exist_ok=True)

BASE_PATH = r"C:\Users\lamin\Documents\maitrise\ASTD\data"
YEAR      = 2019

MONTHS_TO_LOAD = [1, 2, 3]

USECOLS   = "default"
SAMPLING  = [0, -1]

COLS_REQUIRED = [
    "shipid",
    "date_time_utc",
    "latitude",
    "longitude",
    "astd_cat",
    "flagname"
]


In [ ]:


df = tb.load_astd_monthly(
    BASE_PATH, YEAR, months=MONTHS_TO_LOAD, progress=True,
    usecols=USECOLS, sampling=SAMPLING, remove_nan_rows=COLS_REQUIRED
)





c:\Users\lamin\miniconda3\envs\torch-gpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading ASTD CSVs: 100%|██████████| 3/3 [03:49<00:00, 76.40s/it]


In [3]:
df.head()

,shipid,date_time_utc,flagname,iceclass,astd_cat,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude
0,162,2019-01-01 00:00:00+00:00,Iceland,NaN,Unknown,NaN,2.242847,209,-21.827280,64.134209
1,1645,2019-01-01 00:00:00+00:00,Iceland,NaN,Fishing vessels,< 1000 GT,0.731465,361,-23.703033,64.896217
2,3619,2019-01-01 00:00:00+00:00,Norway,NaN,Other activities,1000 - 4999 GT,491.440918,369,14.545295,68.205574
3,2490,2019-01-01 00:00:00+00:00,Iceland,NaN,Fishing vessels,1000 - 4999 GT,2.241636,82,-23.127956,66.069336
4,12235,2019-01-01 00:00:01+00:00,Bahamas,NaN,Crude oil tankers,50000 - 99999,929.041382,361,16.836599,70.436508


In [4]:
tracks = tb.build_ship_tracks(df,
                              max_time_gap_hours=25,
                              max_distance_km=400,
                              min_track_length=1,
                              matching_strategy="balanced",
                              )

Data after cleaning:
  Date range: 2019-01-01 00:00:00+00:00 to 2019-03-31 23:59:58+00:00
  Ship types: ['unknown' 'fishing vessels' 'other activities' 'crude oil tankers'
 'offshore supply ships' 'general cargo ships'
 'other service offshore vessels' 'passenger ships' 'bulk carriers'
 'ro-ro cargo ships' 'cruise ships' 'refrigerated cargo ships'
 'chemical tankers' 'oil product tankers' 'container ships' 'gas tankers']
  Unique ships: 7100
Creating segments for 7100 unique shipids
Created 7100 segments
Sample segment: unknown|iceland|nan|nan


C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:240: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = (ship_means.groupby('astd_cat')


In [4]:
tracks = tb.build_ship_tracks(df,
                              max_time_gap_hours=25,
                              max_distance_km=400,
                              min_track_length=1,
                              matching_strategy="balanced",
                              )

Data after cleaning:
  Date range: 2019-01-01 00:00:00+00:00 to 2019-03-31 23:59:58+00:00
  Ship types: ['fishing vessels' 'unknown' 'other activities' 'crude oil tankers'
 'offshore supply ships' 'general cargo ships'
 'other service offshore vessels' 'passenger ships' 'bulk carriers'
 'ro-ro cargo ships' 'cruise ships' 'refrigerated cargo ships'
 'chemical tankers' 'oil product tankers' 'container ships' 'gas tankers']
  Unique ships: 13246
Creating segments for 13246 unique shipids
Created 13246 segments
Sample segment: fishing vessels|iceland|nan|< 1000 gt


C:\Users\lamin\Documents\maitrise\ASTD\TrackBuilder\track_builder\core\track_helpers.py:240: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  out = (ship_means.groupby('astd_cat')


In [5]:
# head of tracks
tracks.head()

,month,segment_id,track_id
0,2019-01,162,1
1,2019-01,1645,2
2,2019-01,3619,3
3,2019-01,2490,4
4,2019-01,12235,5


In [6]:
tb.get_track_statistics(tracks, df)

{'n_tracks': 6432,
 'n_segments': 7100,
 'avg_length': 1.103855721393035,
 'max_length': 3,
 'lengths': track_id
 1       1
 2       1
 3       1
 4       1
 5       1
        ..
 6428    1
 6429    1
 6430    1
 6431    1
 6432    1
 Length: 6432, dtype: int64,
 'by_month': month
 2019-01    3829
 2019-02    2096
 2019-03    1175
 dtype: int64,
 'by_ship_type': astd_cat
 fishing vessels                   1658
 bulk carriers                     1195
 other activities                   950
 general cargo ships                838
 passenger ships                    547
 container ships                    514
 unknown                            378
 chemical tankers                   257
 ro-ro cargo ships                  210
 crude oil tankers                  122
 offshore supply ships              115
 refrigerated cargo ships           107
 oil product tankers                 71
 gas tankers                         58
 other service offshore vessels      44
 cruise ships             

In [ ]:
tracks.to_parquet(os.path.join(output_path, "tracks_2019_Q2.parquet"))


In [8]:
# tracks that span at least 2 months
month_counts = tracks.groupby("track_id")["month"].nunique()
multi_month_ids = month_counts[month_counts >= 2].index

print("Number of tracks spanning at least 2 months:", len(multi_month_ids))
tracks[tracks["track_id"].isin(multi_month_ids)].head(10)


Number of tracks spanning at least 2 months: 592


,month,segment_id,track_id
8,2019-01,5966,9
9,2019-02,2903,9
19,2019-01,171,19
20,2019-02,161,19
21,2019-03,262,19
27,2019-01,1541,25
28,2019-02,1418,25
38,2019-01,9019,35
39,2019-02,5670,35
45,2019-01,5901,41


In [9]:
# tracks that span at least 3 months
month_counts = tracks.groupby("track_id")["month"].nunique()
multi_month_ids = month_counts[month_counts >= 3].index

print("Number of tracks spanning at least 3 months:", len(multi_month_ids))
tracks[tracks["track_id"].isin(multi_month_ids)].head(10)


Number of tracks spanning at least 3 months: 76


,month,segment_id,track_id
19,2019-01,171,19
20,2019-02,161,19
21,2019-03,262,19
63,2019-01,693,58
64,2019-02,708,58
65,2019-03,740,58
71,2019-01,660,64
72,2019-02,672,64
73,2019-03,703,64
87,2019-01,4490,76


In [10]:
df = df.copy()
df["month"] = pd.to_datetime(df["date_time_utc"]).dt.strftime("%Y-%m")

df_with_tracks = df.merge(
    tracks,
    left_on=["month", "shipid"],
    right_on=["month", "segment_id"],
    how="inner",
)

fig = tb.plot_ship_tracks(
    df_with_tracks,
    color_by="track_id",
    color_mode="categorical",
    show_points=False,
    map_style="open-street-map",
    title="Tracks for first 3 months of 2019",
)

fig.update_layout(showlegend=False)

#  export to HTML
tb.export_figure(fig, "tracks_3months.html")

In [12]:
# Light version: limit the number of displayed tracks + sampling

import numpy as np
import pandas as pd

# df_tracks must contain positions + track_id for the 3 months
work = df_with_tracks.copy()

print("Initial size:", len(work), "lines")

#  Limit the number of displayed tracks (max 200)
max_tracks = 200
all_ids = work["track_id"].dropna().unique()

if len(all_ids) > max_tracks:
    np.random.seed(42)
    keep_ids = np.random.choice(all_ids, size=max_tracks, replace=False)
    work = work[work["track_id"].isin(keep_ids)]
else:
    keep_ids = all_ids

print(f"Tracks displayed: {len(keep_ids)} / {len(all_ids)}")

#  Sampling: 1 point out of 10 per track
work = (
    work.sort_values(["track_id", "date_time_utc"])
        .groupby("track_id", group_keys=False)
        .apply(lambda g: g.iloc[::10])
)

print("sampling kept :", len(work), "lines")



# Keep only the truly Arctic zone
work = work.query("latitude >= 60 and longitude >= -80 and longitude <= 40")


# delete big jumps in position (e.g., due to track merging)
def remove_big_jumps(g, max_deg=10):
    g = g.sort_values("date_time_utc")
    dlat = g["latitude"].diff().abs()
    dlon = g["longitude"].diff().abs()
    # we keep the first point (diff is NaN) and points where the jump is <= max_deg
    mask = dlat.isna() | ((dlat <= max_deg) & (dlon <= max_deg))
    return g[mask]

work = work.groupby("track_id", group_keys=False).apply(remove_big_jumps)


fig = tb.plot_ship_tracks(
    work,
    color_by="track_id",
    color_mode="categorical",
    show_points=True,               
    map_style="open-street-map",
    title="Tracks for first 3 months of 2019 (light version)"
)

fig.update_layout(showlegend=False)

fig.show()

fig.write_html("tracks_3month_light.html", include_plotlyjs='cdn')


Initial size: 1178313 lines
Tracks displayed: 200 / 6432
sampling kept : 3203 lines


C:\Users\lamin\AppData\Local\Temp\ipykernel_28284\2780595869.py:28: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

C:\Users\lamin\AppData\Local\Temp\ipykernel_28284\2780595869.py:48: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [5]:
tracks = pd.read_parquet(os.path.join(output_path, "tracks_2019_Q2.parquet"))

tracks

,month,segment_id,track_id
0,2019-01,162,1
1,2019-01,1645,2
2,2019-01,3619,3
3,2019-01,2490,4
4,2019-01,12235,5
...,...,...,...
7095,2019-03,6483,6428
7096,2019-03,20626,6429
7097,2019-03,20619,6430
7098,2019-03,20546,6431


In [11]:
df_track_19 = tb.load_positions_for_track(track_id=19, track_table=tracks, base_path=BASE_PATH, chunksize=100000)
df_track_64 = tb.load_positions_for_track(track_id=64, track_table=tracks, base_path=BASE_PATH, chunksize=100000)
df_track_76 = tb.load_positions_for_track(track_id=76, track_table=tracks, base_path=BASE_PATH, chunksize=100000)


Loading positions for track 76: 100%|██████████| 3/3 [01:33<00:00, 31.33s/it]


In [14]:
track = 76
fig_test = tb.plot_individual_track(
                track,
                tracks,
                df_track_76,
                show_segments=True,
                map_style="open-street-map",
                title=f"Track {track} ",
            )

fig_test.show()
# fig_test.write_html("fig_58.html", include_plotlyjs='cdn')